# AQIS lab: Sprint 2 live stack, Colab part (data prep, TEN VAD on x86, emotion models)

Runs the Colab stages of WP-1.2 (`docs/specs/sprints/WP-1.2-S2-live-stack.md`): mount Drive, clone the private repository at the tag `AQIS_TAG`, install the locked environment, run `python -m aqis_lab.live <action>`, write the unit log, and free the runtime. The VAD (all but TEN VAD), turn-end and streaming stages run on the Oracle Arm VM (`docs/oracle/S2_RUNBOOK.md`), because the brief measures CPU speed there. TEN VAD has no Arm build, so its `vad` run is the `vad` stage below, on a Colab CPU runtime (x86_64).

**Before you start:**
- Sprint 1 `prep` must have run: `MyDrive/AQIS_LAB/s1/data/` holds `us_phone`, `nonspeech`, `anchor` and `us_conv`. Prep stops naming any file it cannot find.
- Colab Secrets: `HF_TOKEN` (read) and `GITHUB_TOKEN`, as in Sprint 1. Drive: `MyDrive/AQIS_LAB` needs about 6 GB free.

**Each session:** edit only cell 1 (`STAGE`, `MODELS`), pick the runtime below, then **Run all**. The last cell writes the unit log and, when every action exited 0, frees the runtime. After a failure it stays connected: disconnect it yourself when done.

| STAGE | Runtime | What it does |
| --- | --- | --- |
| `prep` | CPU | builds the vad, turn, stream and emotion test sets from Sprint 1 data, HarperValleyBank and CREMA-D (resumable: finished files are skipped) |
| `vad` | CPU | **only `ten_vad`**, on all five VAD conditions, with the same smoothing as Silero v5 (64 ms speech, 100 ms silence). It installs `libc++1` first and never reruns the other VADs. Results go to `AQIS_LAB/s2/results/vad/`, beside the Oracle's |
| `smoke` | **L4 GPU** | every emotion model on 3 clips of the first condition |
| `emotion` | **L4 GPU** | every emotion model on all clips in clean, phone, webrtc and noise_10 (rerun after a dropped session: finished files are skipped) |
| `report` | CPU | number tables, `scorecard.md`, plots and `report/s2_report.zip` from every result file on Drive, including the Oracle's |

Run the stages in this order: `prep`, then the Oracle stages, `vad` (TEN VAD), `smoke`, `emotion`, `report`. For `vad`, set `VAD_LIMIT = 3` first for a smoke run, then `100` for a pilot, then `None`. Use an L4; the emotion stage should take about 20 minutes.

In [ ]:
# 1) Settings: edit STAGE and MODELS only. Then Google Drive is mounted.
AQIS_TAG = "s2-colab-1"
STAGE = "prep"          # prep | vad | smoke | emotion | report
MODELS = ["emotion2vec_seed", "emotion2vec_base", "wav2vec2_er", "acoustic_baseline"]     # emotion models; the vad stage ignores this
VAD_LIMIT = None        # vad stage only: clips per file (3 = smoke, 100 = pilot, None = all)

REPO = "https://github.com/NasirAman-promcare/voice-bot-vad-harness.git"
WORKDIR = "/content/aqis"
SCRATCH = "/content/s2_work"
DRIVE_MOUNT = "/content/drive"
AQIS_DRIVE = "/content/drive/MyDrive/AQIS_LAB"
S1_OUT = AQIS_DRIVE + "/s1"
S2_OUT = AQIS_DRIVE + "/s2"

STAGES = ("prep", "vad", "smoke", "emotion", "report")
assert STAGE in STAGES, f"STAGE must be one of {STAGES}"

from google.colab import drive

drive.mount(DRIVE_MOUNT)

In [ ]:
# 2) Clone the private repository at AQIS_TAG. The GITHUB_TOKEN secret is read inside this one git command's
#    environment; it is never stored in a variable, a URL, .git/config or any output.
import base64
import os
import subprocess

from google.colab import userdata

subprocess.run(["rm", "-rf", WORKDIR], check=True)
subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", AQIS_TAG, REPO, WORKDIR],
               env=dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                        GIT_CONFIG_VALUE_0="Authorization: Basic " + base64.b64encode(
                            ("x-access-token:" + userdata.get("GITHUB_TOKEN")).encode()).decode()),
               check=True)

In [ ]:
# 3) Environments. The lab's own .venv (hash-checked phase 0 lock) is always built: it runs the unit log. prep and report
#    also get the s1-prep lock on top (the Sprint 1 prep code is reused). vad gets a CPU venv from the hash-pinned s2-live
#    lock (after installing libc++1, which the TEN VAD library needs); smoke and emotion get a GPU venv from the same lock. A failed install is printed and recorded in SETUP_ERROR, so the last cell still frees the runtime.
import platform
import re

PY = WORKDIR + "/.venv/bin/python"
LOCKS = WORKDIR + "/requirements/aqis_lab"
LIVE_PY = PY
SETUP_ERROR = None
try:
    subprocess.run(["pip", "install", "--quiet", "uv==0.8.17"], check=True)
    subprocess.run(["bash", "aqis_lab/deploy/dev_bootstrap.sh"], cwd=WORKDIR, check=True)
    if STAGE in ("prep", "report"):
        subprocess.run(["uv", "pip", "install", "--python", PY, "--require-hashes", "--build-constraints",
                        LOCKS + "/phase0.txt", "-r", LOCKS + "/s1-prep.txt"], check=True)
    else:
        if STAGE == "vad":
            assert platform.machine() == "x86_64", "TEN VAD has a library for Linux x86_64 only; run this stage on a Colab CPU runtime"
            subprocess.run("apt-get update && apt-get install -y libc++1", shell=True, check=True)
        else:
            print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                 capture_output=True, text=True, check=True).stdout.strip())
        probe = open(WORKDIR + "/docs/environment/colab_probe.txt").read()
        colab_python = re.search(r"Python Version:\s*(\d+\.\d+)", probe).group(1)
        subprocess.run(["uv", "venv", "--clear", "--python", colab_python, "/content/venv-live"], check=True)
        subprocess.run(["uv", "pip", "sync", "--python", "/content/venv-live/bin/python", "--require-hashes",
                        "--build-constraints", LOCKS + "/phase0.txt", LOCKS + "/s2-live.txt"], check=True)
        LIVE_PY = "/content/venv-live/bin/python"
except (subprocess.CalledProcessError, AssertionError) as exc:
    SETUP_ERROR = repr(exc)
    print("SETUP FAILED:", SETUP_ERROR)

In [ ]:
# 4) Unit log start (the runtime is billed from here), then the stage. HF_TOKEN goes into each subprocess's environment
#    only. The output streams line by line. The last cell frees the runtime only if every action exits 0.
env = dict(os.environ, HF_TOKEN=userdata.get("HF_TOKEN"), PYTHONUNBUFFERED="1", PYTHONPATH=WORKDIR, MPLBACKEND="Agg")
results = {}


def live(python, action, *extra):
    """Run one aqis_lab.live action and stream its output; returns the exit code."""
    process = subprocess.Popen([python, "-u", "-m", "aqis_lab.live", action, "--out", S2_OUT, "--work-dir", SCRATCH, *extra],
                               cwd=WORKDIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    return process.wait()


if SETUP_ERROR:
    print("Skipping the stage because setup failed:", SETUP_ERROR)
else:
    subprocess.run([PY, "-m", "aqis_lab", "unitlog", "start", "--drive-mount", DRIVE_MOUNT], cwd=WORKDIR, check=True)
    if STAGE == "prep":
        results[STAGE] = live(PY, "prep", "--s1-out", S1_OUT)
    elif STAGE == "vad":
        # Only TEN VAD: silero_v5, silero_v4, firered_vad and webrtc_vad were run on the Oracle VM and are not rerun here.
        limit = ["--limit", str(VAD_LIMIT)] if VAD_LIMIT else []
        results[STAGE] = live(LIVE_PY, "vad", "--models", "ten_vad", *limit)
    elif STAGE == "report":
        results[STAGE] = live(PY, "report")
    elif STAGE == "smoke":
        results[STAGE] = live(LIVE_PY, "emotion", "--models", *MODELS, "--conditions", "clean", "--limit", "3", "--device", "cuda")
    else:
        results[STAGE] = live(LIVE_PY, "emotion", "--models", *MODELS, "--device", "cuda")
    print("exit codes (0 = done; 2 = error):")
    for name, rc in results.items():
        print(f"  {name:28s} {rc}")

In [ ]:
# 5) End of session: append this session's estimated compute units to AQIS_LAB/logs/units.csv. The runtime is freed
#    (so the GPU stops costing units) only when setup worked and every action above exited 0; after any failure it stays
#    connected so you can read the output and debug. Then free it yourself: Runtime > Disconnect and delete runtime.
try:
    subprocess.run([PY, "-m", "aqis_lab", "unitlog", "end", "--drive-mount", DRIVE_MOUNT], cwd=WORKDIR, check=True)
except subprocess.CalledProcessError as exc:
    print("UNIT LOG NOT WRITTEN, note this session's hours by hand:", exc)
if SETUP_ERROR is None and results and all(rc == 0 for rc in results.values()):
    from google.colab import runtime

    runtime.unassign()
else:
    print("NOT freeing the runtime: setup failed or an action exited non-zero (see the exit codes above). "
          "A GPU runtime keeps costing units until you disconnect it.")